In [ ]:
# ============================================================================
# CELL 1: Setup Google Colab Environment
# Run this cell FIRST
# ============================================================================

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Setup complete!")

Mounted at /content/drive
✓ Setup complete!


In [ ]:
# Install required packages
!pip install -q openai tqdm
!pip install pyarrow

In [ ]:
# ============================================================================
# CELL 2: Import Libraries and Configuration
# ============================================================================

import pandas as pd
import numpy as np
import json
import time
import os
from datetime import datetime
from typing import Dict, List, Optional
from tqdm.auto import tqdm
from openai import OpenAI
import openai
import re
import hashlib

# Configuration
INPUT_FILE = "/content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/forum_enriched.csv"
OUTPUT_FILE = "/content/drive/MyDrive/PropInsight/labeled/multi_forum_property_posts_hwz_processed_2023_2025/forum_labeled.csv"


# Checkpoint paths (FIX)
CHECKPOINT_FILE = "/content/drive/MyDrive/PropInsight/labeled/checkpoint_hwz/multi_forum_property_posts_hwz_processed_2023_2025/labeling_checkpoint.csv"

# Make the *directory*, not the file path
CHECKPOINT_DIR = os.path.dirname(CHECKPOINT_FILE)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# If a directory was mistakenly created at the file path, auto-repair it
if os.path.isdir(CHECKPOINT_FILE):
    # Move that mistaken directory aside so we can use the file path properly
    repaired_dir = CHECKPOINT_FILE + "_backup_dir"
    print(f"⚠ Found a directory at checkpoint file path. Moving it to: {repaired_dir}")
    os.rename(CHECKPOINT_FILE, repaired_dir)


# Make the *directory*, not the file path
OUTPUT_DIR = os.path.dirname(OUTPUT_FILE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# If a directory was mistakenly created at the file path, auto-repair it
if os.path.isdir(OUTPUT_FILE):
    # Move that mistaken directory aside so we can use the file path properly
    repaired_dir = OUTPUT_FILE + "_backup_dir"
    print(f"⚠ Found a directory at checkpoint file path. Moving it to: {repaired_dir}")
    os.rename(OUTPUT_FILE, repaired_dir)





# OpenAI Settings
OPENAI_MODEL = "gpt-4o"  # Cost-effective and fast
MAX_RETRIES = 3
RETRY_DELAY = 2
BATCH_SIZE = 10  # Save checkpoint every 10 posts
RATE_LIMIT_DELAY = 1  # 1 second between API calls

print("✓ Configuration loaded")

✓ Configuration loaded


In [ ]:
# ============================================================================
# CELL 3: Enter Your OpenAI API Key
# Get your key from: https://platform.openai.com/api-keys
# ============================================================================

from google.colab import userdata

try:
    # Get API key from Colab secrets
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

    # Initialize OpenAI client
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("✓ OpenAI API key loaded from Colab secrets")
    print("✓ OpenAI client initialized")

    # Test the connection
    test_response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": "test"}],
        max_tokens=5
    )
    print("✓ API connection successful!")
    print(f"✓ Using model: {OPENAI_MODEL}")

except Exception as e:
    print("✗" * 70)
    print("ERROR: Could not load OpenAI API key from Colab secrets")
    print("✗" * 70)
    print(f"\nError details: {str(e)}")
    print("\nTo fix this:")
    print("1. Click the 🔑 key icon in the left sidebar")
    print("2. Click 'Add new secret'")
    print("3. Name: OPENAI_API_KEY")
    print("4. Value: <paste your API key>")
    print("5. Toggle 'Notebook access' ON")
    print("6. Re-run this cell")
    print("\nGet your API key from: https://platform.openai.com/api-keys")
    raise


✓ OpenAI API key loaded from Colab secrets
✓ OpenAI client initialized
✓ API connection successful!
✓ Using model: gpt-4o


In [ ]:
# ============================================================================
# CELL 4: Define Helper Functions
# ============================================================================

def extract_source_from_url(url: str) -> str:
    """Extract forum source from URL."""
    if pd.isna(url):
        return "unknown"
    url_lower = url.lower()
    if "hardwarezone" in url_lower or "hwz" in url_lower:
        return "HWZ"
    elif "singaporeexpats" in url_lower or "sgexpats" in url_lower:
        return "SGExpats"
    return "unknown"


def create_llm_prompt(text: str, has_singlish: bool, policy_flags: Dict) -> str:
    """Create prompt for OpenAI to analyze the forum post."""

    policy_info = []
    for policy, flag in policy_flags.items():
        if flag:
            policy_name = policy.replace('flag_', '')
            policy_info.append(policy_name)

    policy_context = ", ".join(policy_info) if policy_info else "None"

    prompt = f"""Analyze this Singapore property forum post and extract sentiment and context.

POST TEXT:
{text[:2000]}

EXISTING CONTEXT:
- Singlish detected: {has_singlish}
- Policies mentioned: {policy_context}

Extract these fields in JSON format:

1. overall_sentiment: Overall market optimism ("positive", "neutral", "negative")
2. price_sentiment: Price direction perception ("rising", "neutral", "falling")
3. policy_sentiment: Attitude toward government policies ("positive", "neutral", "negative")
4. affordability_sentiment: Affordability perception ("positive", "neutral", "negative")
5. location: Singapore locations mentioned (e.g., "Woodlands, D19" or "none")
6. policy_mentioned: Policy terms found (e.g., "ABSD, TDSR" or "none")
7. cultural_context: Singaporean cultural meaning (e.g., "kiasu urgency" or "none")
8. emotion: Dominant emotion ("joy", "anger", "fear", "trust", "anticipation", "surprise", "sadness", "disgust", "neutral")

IMPORTANT:
- Be specific with locations (neighborhood names, districts)
- Policy sentiment = attitude toward government intervention
- Affordability sentiment = whether poster thinks it's affordable
- Cultural context = uniquely Singaporean perspectives (kiasu, pragmatic, FOMO, etc.)

Return ONLY valid JSON:
{{
  "overall_sentiment": "",
  "price_sentiment": "",
  "policy_sentiment": "",
  "affordability_sentiment": "",
  "location": "",
  "policy_mentioned": "",
  "cultural_context": "",
  "emotion": ""
}}"""

    return prompt


def call_openai_api(client: OpenAI, prompt: str) -> Dict:
    """Call OpenAI API with retry logic."""
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": "You are an expert in analyzing Singapore property forum discussions. You understand Singlish, local context, and market sentiment. Respond with valid JSON only."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,
                max_tokens=500,
                response_format={"type": "json_object"}
            )

            result = json.loads(response.choices[0].message.content)
            return result

        except openai.RateLimitError:
            wait_time = RETRY_DELAY * (attempt + 1)
            print(f"⚠ Rate limit. Waiting {wait_time}s...")
            time.sleep(wait_time)

        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                print(f"⚠ Error (attempt {attempt+1}): {str(e)}")
                time.sleep(RETRY_DELAY)
            else:
                print(f"✗ Failed after {MAX_RETRIES} attempts")

    # Return defaults if all attempts fail
    return {
        "overall_sentiment": "neutral",
        "price_sentiment": "neutral",
        "policy_sentiment": "neutral",
        "affordability_sentiment": "neutral",
        "location": "none",
        "policy_mentioned": "none",
        "cultural_context": "none",
        "emotion": "neutral"
    }




def process_single_post(row: pd.Series, client: OpenAI) -> Dict:
    """Process one forum post and return all labels."""

    # Extract existing data
    text = str(row.get('clean_text', ''))
    has_singlish = bool(row.get('has_singlish', False))

    # Get policy flags
    policy_flags = {
        'flag_ABSD': row.get('flag_ABSD', False),
        'flag_BSD': row.get('flag_BSD', False),
        'flag_SSD': row.get('flag_SSD', False),
        'flag_TDSR': row.get('flag_TDSR', False),
        'flag_LTV': row.get('flag_LTV', False),
        'flag_MSR': row.get('flag_MSR', False),
        'flag_HLE': row.get('flag_HLE', False)
    }

    # Call OpenAI API
    prompt = create_llm_prompt(text, has_singlish, policy_flags)
    llm_result = call_openai_api(client, prompt)

    # Combine results with simple derived fields
    labels = {
        'overall_sentiment': llm_result.get('overall_sentiment', 'neutral'),
        'price_sentiment': llm_result.get('price_sentiment', 'neutral'),
        'policy_sentiment': llm_result.get('policy_sentiment', 'neutral'),
        'affordability_sentiment': llm_result.get('affordability_sentiment', 'neutral'),
        'location': llm_result.get('location', 'none'),
        'policy_mentioned': llm_result.get('policy_mentioned', 'none'),
        'singlish_detected': has_singlish,
        'cultural_context': llm_result.get('cultural_context', 'none'),
        'emotion': llm_result.get('emotion', 'neutral'),
        'source': extract_source_from_url(row.get('thread_url', ''))
    }


    # ========= START UPDATE: normalize labels (append inside process_single_post) =========

    # Normalize label values (avoid typos/odd strings)
    def _clamp_set(val, allowed, default="neutral"):
        val = (str(val or "")).lower().strip()
        return val if val in allowed else default

    labels['overall_sentiment']       = _clamp_set(labels['overall_sentiment'],       {"positive","neutral","negative"})
    labels['price_sentiment']         = _clamp_set(labels['price_sentiment'],         {"rising","neutral","falling"})
    labels['policy_sentiment']        = _clamp_set(labels['policy_sentiment'],        {"positive","neutral","negative"})
    labels['affordability_sentiment'] = _clamp_set(labels['affordability_sentiment'], {"positive","neutral","negative"})
    labels['emotion']                 = _clamp_set(labels['emotion'],                 {"joy","anger","fear","trust","anticipation","surprise","sadness","disgust","neutral"})

    # Tidy simple strings
    labels['location']         = (labels.get('location') or 'none').strip()
    labels['policy_mentioned'] = (labels.get('policy_mentioned') or 'none').strip()

# ========= END UPDATE =========


    return labels

print("✓ Helper functions defined")

✓ Helper functions defined


In [ ]:
# ============================================================================
# CELL 5: Load Data and Initialize
# ============================================================================

print("Loading forum data...")
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"✓ Loaded {len(df)} forum posts")
    print(f"  Columns: {len(df.columns)}")
    print(f"  First few columns: {list(df.columns)[:5]}")
except FileNotFoundError:
    print(f"✗ File not found: {INPUT_FILE}")
    print("Please check the path and try again.")
    raise
except Exception as e:
    print(f"✗ Error: {str(e)}")
    raise

# ========= START UPDATE: Build 'body' and prefilter low-signal =========

# Build a unified 'body' column (title + text) so downstream code is consistent
if "body" not in df.columns:
    title_col = next((c for c in ["title","thread_title","subject"] if c in df.columns), None)
    text_col  = next((c for c in ["clean_text","text","content","post_text"] if c in df.columns), None)

    # Ensure we are operating on Series, even if column doesn't exist
    title_series = df[title_col].astype(str) if title_col and title_col in df.columns else pd.Series([""] * len(df))
    text_series  = df[text_col].astype(str)  if text_col and text_col in df.columns else pd.Series([""] * len(df))

    df["body"] = (title_series + "\n\n" + text_series).str.strip()


# Conservative boilerplate detector
def _is_boiler(s: str) -> bool:
    if not isinstance(s, str) or not s.strip():
        return True
    s = s.lower()
    pats = [
        "skip to main", "share this page", "copyright", "privacy policy",
        "terms of use", "site map", "back to top", "subscribe", "breadcrumbs",
        "last updated", "prevnext"
    ]
    return any(p in s for p in pats)

# Low-signal filter (very conservative): short + no entities/policy OR boilerplate
def _low_signal(row) -> bool:
    txt = str(row.get("body",""))
    length = len(txt)
    no_ent = not str(row.get("entity","") or "").strip()
    no_pol = int(row.get("policy_term_count", 0) or 0) == 0
    return (length < 80 and no_ent and no_pol) or _is_boiler(txt)

_pre_len = len(df)
df = df[~df.apply(_low_signal, axis=1)].copy()
print(f"Prefilter removed {_pre_len - len(df)} low-signal rows; remaining: {len(df)}")

# Reset index after filtering
df.reset_index(drop=True, inplace=True)
print("✓ DataFrame index reset after prefiltering.")

# ========= END UPDATE =========


# Check for checkpoint
print("\nChecking for existing checkpoint...")
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    try:
        checkpoint_df = pd.read_csv(CHECKPOINT_FILE)
        if 'overall_sentiment' in checkpoint_df.columns:
            # Merge checkpoint with current data based on a unique identifier (e.g., hash or original index if preserved)
            # For simplicity and given the reset index, let's assume we need to align based on hash or re-label
            # A more robust approach might involve merging on original identifiers.
            # For now, we'll load the checkpoint and assume we resume from the last labeled row in the checkpoint
            # This might re-label some rows if the prefiltering changes, but is safer than misaligning.
            labeled_count = checkpoint_df['overall_sentiment'].notna().sum()
            if labeled_count > 0:
                print(f"✓ Found checkpoint with {labeled_count} labeled posts")
                # Option 1: Just load the checkpoint and use it as the starting point (simplest, might re-label)
                # df = checkpoint_df # This was the previous problematic approach
                # Option 2: Merge based on hash - requires hash in checkpoint
                if 'hash' in checkpoint_df.columns and 'hash' in df.columns:
                     # Align current df with checkpoint based on hash
                     df = df.set_index('hash').combine_first(checkpoint_df.set_index('hash')).reset_index()
                     # Recalculate start_idx based on non-null 'overall_sentiment' after merge
                     start_idx = df['overall_sentiment'].notna().sum()
                     print(f"  Merged with checkpoint based on hash. Resuming from row {start_idx}")
                else:
                     print("  Checkpoint does not contain 'hash' column for merging. Starting fresh.")
                     start_idx = 0 # Start fresh if cannot merge reliably


    except Exception as e:
        print(f"⚠ Checkpoint exists but couldn't load or merge: {str(e)}")
        print("  Starting fresh...")
        start_idx = 0 # Ensure start_idx is 0 if checkpoint loading/merging fails
else:
    print("  No checkpoint found")

# Initialize label columns if needed
label_columns = [
    'overall_sentiment', 'price_sentiment', 'policy_sentiment',
    'affordability_sentiment', 'location', 'policy_mentioned',
    'singlish_detected', 'cultural_context', 'emotion', 'source'
]

for col in label_columns:
    if col not in df.columns:
        df[col] = None

# ========= START UPDATE: hash-based dedupe checkpoint =========

HASH_PATH = "/content/drive/MyDrive/PropInsight/labeled/checkpoint_shwz/multi_forum_property_posts_hwz_processed_2023_2025/forum_hashes.parquet"
os.makedirs(os.path.dirname(HASH_PATH), exist_ok=True)

def _body_hash(s: str) -> str:
    s = (s or "").strip().lower()
    return hashlib.md5(s.encode("utf-8")).hexdigest()

# Ensure 'hash' column exists after potential checkpoint merge
if "hash" not in df.columns:
    df["hash"] = df["body"].map(_body_hash)


try:
    _seen = pd.read_parquet(HASH_PATH).set_index("hash")
    _seen_set = set(_seen.index)
except Exception:
    _seen_set = set()

# for visibility
_new_to_label = (~df["hash"].isin(_seen_set)).sum()
print(f"New bodies to label (hash-check): {_new_to_label} / {len(df)}")

# ========= END UPDATE =========


print(f"\n✓ Ready to process {len(df) - start_idx} posts")
print(f"  Model: {OPENAI_MODEL}")
print(f"  Checkpoint frequency: every {BATCH_SIZE} posts")

Loading forum data...
✓ Loaded 4 forum posts
  Columns: 13
  First few columns: ['forum_name', 'thread_url', 'post_text', 'date', 'raw_text']
Prefilter removed 4 low-signal rows; remaining: 0
✓ DataFrame index reset after prefiltering.

Checking for existing checkpoint...
  No checkpoint found
New bodies to label (hash-check): 0 / 0

✓ Ready to process 0 posts
  Model: gpt-4o
  Checkpoint frequency: every 10 posts


In [ ]:
# ============================================================================
# CELL 6: Process All Posts (Main Loop)
# ============================================================================

print("=" * 70)
print("Starting labeling process...")
print("=" * 70)
print()

total_posts = len(df)
posts_to_process = total_posts - start_idx

# Progress bar
pbar = tqdm(total=posts_to_process, desc="Labeling", unit="posts")

processed_count = 0
errors = []

try:
    for idx in range(start_idx, total_posts):

        # Skip if already labeled
        #-------- OLD skip (commented; keep for reference) --------
        # if pd.notna(df.at[idx, 'overall_sentiment']):
        #     pbar.update(1)
        #     continue
        # ----------------------------------------------------------

        # ========= START UPDATE: skip if already labeled OR seen by hash =========
        already_labeled = pd.notna(df.at[idx, 'overall_sentiment'])
        already_seen    = df.at[idx, 'hash'] in _seen_set
        if already_labeled or already_seen:
            pbar.update(1)
            continue
        # ========= END UPDATE =========



        try:
            # Process the post
            labels = process_single_post(df.iloc[idx], client)

            # Update dataframe
            for key, value in labels.items():
                df.at[idx, key] = value

            processed_count += 1
            pbar.update(1)

            # Save checkpoint
            # ========= START UPDATE: extend checkpoint with hashes =========
            if processed_count % BATCH_SIZE == 0:
                df.to_csv(CHECKPOINT_FILE, index=False)
                # update hash store with any rows that are newly labeled
                _to_add = df.loc[df['overall_sentiment'].notna(), ['hash']].drop_duplicates()
                if len(_to_add):
                    try:
                        _cur = pd.read_parquet(HASH_PATH)
                    except Exception:
                        _cur = pd.DataFrame(columns=["hash"])
                    _merged = pd.concat([_cur, _to_add], ignore_index=True).drop_duplicates("hash")
                    _merged.to_parquet(HASH_PATH, index=False)

                    # >>> NEW: keep in-memory set in sync so later duplicates in this run are skipped
                    _seen_set.update(set(_to_add['hash']))  # <<< add this line

                pbar.set_postfix({"saved": f"{processed_count} posts", "errors": len(errors)})
            # ========= END UPDATE =========


            # Rate limiting
            time.sleep(RATE_LIMIT_DELAY)

        except KeyboardInterrupt:
            print("\n\n⚠ Interrupted by user")
            print("Saving checkpoint...")
            df.to_csv(CHECKPOINT_FILE, index=False)
            raise

        except Exception as e:
            error_msg = f"Row {idx}: {str(e)}"
            errors.append(error_msg)
            print(f"\n⚠ {error_msg}")
            continue

    pbar.close()

except KeyboardInterrupt:
    print("\nProcess stopped by user. Checkpoint saved.")
    pbar.close()


# ========= START UPDATE: final hash save =========
try:
    _cur = pd.read_parquet(HASH_PATH)
except Exception:
    _cur = pd.DataFrame(columns=["hash"])
_to_add = df.loc[df['overall_sentiment'].notna(), ['hash']].drop_duplicates()
_merged = pd.concat([_cur, _to_add], ignore_index=True).drop_duplicates("hash")
_merged.to_parquet(HASH_PATH, index=False)

# >>> NEW: sync in-memory too (not strictly necessary after loop, but harmless)
_seen_set.update(set(_to_add['hash']))  # <<< add this line
# ========= END UPDATE =========


print()
print("=" * 70)
print("Processing complete!")
print("=" * 70)
print(f"  Total processed: {processed_count}")
print(f"  Errors: {len(errors)}")

if errors:
    print("\nError summary:")
    for err in errors[:5]:  # Show first 5 errors
        print(f"  - {err}")
    if len(errors) > 5:
        print(f"  ... and {len(errors)-5} more")

Starting labeling process...



Labeling: 0posts [00:00, ?posts/s]


Processing complete!
  Total processed: 0
  Errors: 0


In [ ]:
# ============================================================================
# CELL 7: Save Final Output and View Statistics
# ============================================================================

print("\nSaving final labeled dataset...")
try:
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"✓ Saved to: {OUTPUT_FILE}")
    print(f"  Total rows: {len(df)}")
except Exception as e:
    print(f"✗ Error saving: {str(e)}")
    print(f"  Checkpoint available at: {CHECKPOINT_FILE}")

print("\n" + "=" * 70)
print("SUMMARY STATISTICS")
print("=" * 70)

print("\n📊 Overall Sentiment:")
print(df['overall_sentiment'].value_counts())

print("\n📈 Price Sentiment:")
print(df['price_sentiment'].value_counts())

print("\n🏛️ Policy Sentiment:")
print(df['policy_sentiment'].value_counts())

print("\n💰 Affordability Sentiment:")
print(df['affordability_sentiment'].value_counts())

print("\n😊 Emotion Distribution:")
print(df['emotion'].value_counts())

print("\n🗣️ Singlish Detection:")
singlish_count = df['singlish_detected'].sum()
print(f"  Posts with Singlish: {singlish_count} ({singlish_count/len(df)*100:.1f}%)")
print(f"  Posts without: {len(df)-singlish_count} ({(len(df)-singlish_count)/len(df)*100:.1f}%)")

print("\n📍 Top Locations (excluding 'none'):")
locations_df = df[df['location'] != 'none']['location'].str.split(',').explode().str.strip()
top_locations = locations_df.value_counts().head(10)
print(top_locations)

print("\n📋 Policies Mentioned (excluding 'none'):")
policies_df = df[df['policy_mentioned'] != 'none']['policy_mentioned'].str.split(',').explode().str.strip()
top_policies = policies_df.value_counts()
print(top_policies)

print("\n🏠 Source Distribution:")
print(df['source'].value_counts())

print("\n🎭 Cultural Context Examples:")
cultural_examples = df[df['cultural_context'] != 'none']['cultural_context'].value_counts().head(10)
print(cultural_examples)

print("\n" + "=" * 70)
print("✓ All done! 🎉")
print("=" * 70)


Saving final labeled dataset...
✓ Saved to: /content/drive/MyDrive/PropInsight/labeled/multi_forum_property_posts_hwz_processed_2023_2025/forum_labeled.csv
  Total rows: 0

SUMMARY STATISTICS

📊 Overall Sentiment:
Series([], Name: count, dtype: int64)

📈 Price Sentiment:
Series([], Name: count, dtype: int64)

🏛️ Policy Sentiment:
Series([], Name: count, dtype: int64)

💰 Affordability Sentiment:
Series([], Name: count, dtype: int64)

😊 Emotion Distribution:
Series([], Name: count, dtype: int64)

🗣️ Singlish Detection:


ZeroDivisionError: division by zero

In [ ]:

# ============================================================================
# CELL 8: (Optional) View Sample Labeled Posts
# ============================================================================

print("Sample labeled posts:\n")
print("=" * 70)

sample_df = df[df['overall_sentiment'].notna()].sample(min(5, len(df)))

for idx, row in sample_df.iterrows():
    print(f"\n📝 POST #{idx}")
    print(f"Text (excerpt): {str(row['clean_text'])[:200]}...")
    print(f"\nLabels:")
    print(f"  - Overall Sentiment: {row['overall_sentiment']}")
    print(f"  - Price Sentiment: {row['price_sentiment']}")
    print(f"  - Policy Sentiment: {row['policy_sentiment']}")
    print(f"  - Affordability: {row['affordability_sentiment']}")
    print(f"  - Location: {row['location']}")
    print(f"  - Policy Mentioned: {row['policy_mentioned']}")
    print(f"  - Emotion: {row['emotion']}")
    print(f"  - Cultural Context: {row['cultural_context']}")
    print(f"  - Singlish: {'Yes' if row['singlish_detected'] else 'No'}")
    print(f"  - Source: {row['source']}")
    print("-" * 70)

print("\n✓ Done viewing samples!")